# 05 - Like Games Grouping (AI-Attribute Similarity)

## Purpose
For each eligible slot game in the AI-attribute silver table, find the **most similar older games** ("like games") based on AI-extracted game attributes. The neighbours are used downstream in demand forecasting as analogues for new or young games that lack history. A target is included only when `has_eilers_ts_match AND (has_sales_match OR has_rental_match)` is true. Games with both matches appear once, with both flags true. Three independent neighbour lists are produced for every eligible target (regardless of which commercial match the target has):
* **Sales neighbours** - similar games with usable Eilers and internal sales matches.
* **Rental neighbours** - similar games with usable Eilers and rental matches.
* **Eilers neighbours** - similar games that have a match in the Eilers time-series data.

## Input / Output
| Item | Value |
| --- | --- |
| Source table | `development_042_silver_sandbox.demand_forecast.game_attr_ai_silver` |
| Destination table | `development_042_silver_sandbox.demand_forecast.like_games_ai_attr` (overwritten on every run, schema overwrite enabled) |
| Experiment inputs | Set in the cell right after this one: `USER_EXPERIMENT_NAME` (optional label), `PARENT_GAME_PROCESS`, `FEATURE_GROUP_WEIGHTS` |

## Key parameters
| Parameter | Default | Meaning |
| --- | --- | --- |
| `MAX_NEIGHBOUR_DATE` | `2021-06-01` | Only games released **after** this date can be neighbours |
| `MIN_AGE_GAP_MONTHS` | `12` | A neighbour must be at least this many months older than the target game |
| `DISTANCE_METHOD` | `cosine` | Validated as `cosine` or `euclidean`; the ranking logic currently computes cosine distance |
| `FEATURE_GROUP_WEIGHTS` | mechanics, themes, reels, persistence, ways = 1; months_since_release = 0 | Relative importance of each feature group (weight 0 drops the group) |
| `N_SALES_NEIGHBORS` / `N_RENTAL_NEIGHBORS` / `N_EILERS_NEIGHBORS` | `5` / `5` / `5` | Maximum neighbours kept per list |
| `AI_NAME_MODEL` | `system.ai.llama-4-maverick` | Model used via `ai_query` to suggest an experiment name when none is given |

## Workflow by step (steps follow the code cells below, in order)
1. **Configuration** - source/destination table paths, neighbour release-date cutoff and minimum age gap.
2. **Load source** - read the AI-attribute silver table into `ai_attr_silver` and display it.
3. **Setup and feature engineering**
   * Set remaining experiment settings, use the experiment inputs cell values, and generate a unique `EXPERIMENT_ID`.
   * Validate that required columns and match flags (`has_sales_match`, `has_rental_match`, `has_eilers_ts_match`) exist.
   * Extract numeric values from numeric, string, or nested struct columns for **reels**, **ways/lines**, and **persistence**.
   * Collect `theme_*_score` and `mechanic_*_present` (or `mechanic_*_score`) features; missing values become 0.
   * Normalise `parent_own_status` (blank/null -> `__unknown__`), convert to pandas (`pdf`), and deduplicate by normalized `join_game_name_key`. Keep the alphabetically first document's features and merge match flags/names across duplicate documents.
   * Min-max scale features, validate and normalise group weights (they sum to 1), and build a readable experiment name (widget value, else an AI suggestion, else a weight-based fallback).
4. **Preview** - display the prepared pandas dataset `pdf`.
5. **Similarity and neighbour search**
   * Coerce features to numeric, fill theme/mechanic nulls with 0 and other nulls with medians, then min-max scale to [0, 1].
   * Build one weighted feature vector per game. Each group's weight is spread across its columns (`sqrt(group_weight / n_columns)`), so large groups such as themes do not dominate just because they have more columns.
   * For each target game, select eligible candidates: a valid age, at least `MIN_AGE_GAP_MONTHS` older than the target, not the target itself, and released after `MAX_NEIGHBOUR_DATE`.
   * Require an Eilers match and a usable Eilers name for all candidates. Sales candidates also need a sales match and valid sales name; Rental candidates also need a rental match and valid rental name. Rank each list by cosine distance (ties broken by name), and keep the top N.
   * Include only target games with an Eilers flag and at least one Sales/Rental flag. Flags describe source matches, not whether neighbor lists are nonempty. Eilers-only games remain available as performance candidates but are excluded as output targets.
   * Build the Spark DataFrame `like_games_ai_attr_df` with one row per eligible game, source flags and names, the neighbour arrays, and experiment metadata.
6. **Single-game check** - show the ranked Sales, Rental, and Eilers candidates for one target game (default `king khufu dawn`) with the same eligibility rules, for manual validation.
7. **Persist** - overwrite the destination table with `like_games_ai_attr_df`.

## Output schema (`like_games_ai_attr`)
| Column | Description |
| --- | --- |
| `join_game_name` | Target game key |
| `to_sales_name_match`, `to_rental_name_match`, `to_eilers_ts_name_match` | Target's matched source names |
| `has_sales_match`, `has_rental_match`, `has_eilers_ts_match` | Boolean source match flags; both Sales and Rental can be true |
| `sales_rental_match_type` | `sales`, `rental`, or `both` |
| `game_category`, `own_status`, `parent_own_status`, `game_classification` | Target descriptors (for information only; they do not restrict matching) |
| `max_months_since_release`, `game_release_date` | Target age and release date |
| `sales_similar_games` | Array of structs: `join_game_name_key`, `matched_source_name`, `similarity_score` (1 - distance, floored at 0), `distance` |
| `rental_similar_games` | Same structure, with `matched_source_name` from `to_rental_name_match` |
| `eilers_similar_games` | Same structure, for Eilers-matched neighbours |
| `experiment_readable_name`, `experiment_id`, `distance_method`, `feature_group_weights` | Run metadata for comparing experiments |

## Notes
* Step 5 needs variables from step 3 (`pdf`, `active_groups`, `normalized_weights`, name columns, experiment metadata), and step 6 needs helpers from step 5. Run the cells in order.
* Because the minimum age gap is enforced, the newest games have the most candidate neighbours and the oldest games may have none.
* `months_since_release` has weight 0 by default, so release age is used only as an eligibility filter, not as a similarity feature.
* Eligible targets are kept even if no neighbors satisfy the date/age requirements; their arrays are empty. A missing target age also produces empty arrays.
* Source flags control target inclusion. Neighbor selection additionally requires usable matched names; blank/null/None names cannot be selected for that source.
* Setting `DISTANCE_METHOD = "euclidean"` passes validation, but the distance function always computes cosine distance.

In [0]:
# -----------------------------
# Experiment inputs (edit here, then run all cells below)
# -----------------------------
OVERWRITE=False
# Optional readable name. Leave blank to get an AI-suggested / weight-based name.
USER_EXPERIMENT_NAME = "p2"
expid=1
# One of: "p1_parent_games_only", "p2_current", "p3_include_parent"
 

# Relative importance of each feature group (0 drops the group).
FEATURE_GROUP_WEIGHTS = {
    "mechanics": 1.0,
    "themes": 1.0,
    "reels": 1.0,
    "persistence": 1.0,
    "ways": 1.0,
    "months_since_release": 0.0,
}

# Validation
PARENT_GAME_PROCESSES = [
    "p1_parent_games_only",
    "p2_current",
    "p3_include_parent",
]
PARENT_GAME_PROCESS = PARENT_GAME_PROCESSES[expid]
USER_EXPERIMENT_NAME = (USER_EXPERIMENT_NAME or "").strip()
PARENT_GAME_PROCESS = PARENT_GAME_PROCESS.strip().lower()
if PARENT_GAME_PROCESS not in PARENT_GAME_PROCESSES:
    raise ValueError(
        f"Invalid PARENT_GAME_PROCESS: {PARENT_GAME_PROCESS!r}. "
        f"Expected one of {PARENT_GAME_PROCESSES}."
    )


In [0]:
ai_attr_silver_path='development_042_silver_sandbox.demand_forecast.game_attr_ai_silver'
like_games_dest='development_042_silver_sandbox.demand_forecast.like_games_ai_attr'
MAX_NEIGHBOUR_DATE="2021-06-01"
MIN_AGE_GAP_MONTHS = 12


In [0]:
ai_attr_silver=spark.table(ai_attr_silver_path)
display(ai_attr_silver)

In [0]:
import re
import uuid
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, NumericType, StringType


# -----------------------------
# Experiment settings
# -----------------------------

DISTANCE_METHOD = "cosine"  # "cosine" or "euclidean"

# USER_EXPERIMENT_NAME, PARENT_GAME_PROCESS and FEATURE_GROUP_WEIGHTS
# are set in the experiment inputs cell at the top of the notebook.

N_SALES_NEIGHBORS = 5
N_RENTAL_NEIGHBORS = 5
N_EILERS_NEIGHBORS = 5
AI_NAME_MODEL = "system.ai.llama-4-maverick"

EXPERIMENT_ID = f"aiattr_{uuid.uuid4().hex[:10]}"

method = DISTANCE_METHOD.strip().lower()
if method not in {"cosine", "euclidean"}:
    raise ValueError("DISTANCE_METHOD must be 'cosine' or 'euclidean'.")


# -----------------------------
# Inspect source schema
# -----------------------------

source_df = ai_attr_silver
source_columns = {field.name: field for field in source_df.schema.fields}

OWNED_STATUS_COL = (
    "own_status" if "own_status" in source_columns
    else "owned_status" if "owned_status" in source_columns
    else None
)

required_columns = [
    "game_name",
    "parent_game",
    "to_sales_name_match",
    "to_rental_name_match",
    "game_release_date",
    "game_category",
    "join_game_name_key",
    "parent_own_status",
    "game_classification",
    "max_months_since_release",
]

if OWNED_STATUS_COL is None:
    raise ValueError("Could not find own_status or owned_status in ai_attr_silver.")

missing_columns = [c for c in required_columns if c not in source_columns]
if missing_columns:
    raise ValueError(
        f"Missing required columns in ai_attr_silver: {missing_columns}"
    )

for flag_col in ["has_sales_match", "has_rental_match", "has_eilers_ts_match"]:
    if flag_col not in source_columns:
        raise ValueError(f"Missing required match flag: {flag_col}")


# -----------------------------
# Helpers for extracting numeric values
# -----------------------------

def extract_numeric_from_string(value_expr):
    """Extract the first integer or decimal from a string; return null if absent."""
    extracted = F.regexp_extract(
        value_expr.cast("string"),
        r"(-?\d+(?:\.\d+)?)",
        1,
    )

    return F.when(
        F.length(extracted) > 0,
        extracted.cast("double"),
    ).otherwise(F.lit(None).cast("double"))


def get_leaf_fields(data_type, prefix=()):
    """Return scalar leaf paths for fields inside nested structs."""
    leaves = []

    if isinstance(data_type, StructType):
        for field in data_type.fields:
            leaves.extend(
                get_leaf_fields(field.dataType, prefix + (field.name,))
            )
    else:
        leaves.append((prefix, data_type))

    return leaves


def numeric_feature_expr(root_col, preferred_names):
    """Extract a numeric feature from a numeric, string, or struct column."""
    data_type = source_columns[root_col].dataType

    if isinstance(data_type, NumericType):
        return F.col(root_col).cast("double")

    if isinstance(data_type, StringType):
        return extract_numeric_from_string(F.col(root_col))

    if isinstance(data_type, StructType):
        leaves = get_leaf_fields(data_type)
        chosen = None

        # Prefer fields with names that commonly contain the extracted value.
        for preferred_name in preferred_names:
            chosen = next(
                (
                    (path, leaf_type)
                    for path, leaf_type in leaves
                    if path[-1].lower() == preferred_name
                    and isinstance(leaf_type, (NumericType, StringType))
                ),
                None,
            )
            if chosen:
                break

        # Otherwise use the first numeric leaf, then the first string leaf.
        if chosen is None:
            chosen = next(
                (
                    (path, leaf_type)
                    for path, leaf_type in leaves
                    if isinstance(leaf_type, NumericType)
                ),
                None,
            )

        if chosen is None:
            chosen = next(
                (
                    (path, leaf_type)
                    for path, leaf_type in leaves
                    if isinstance(leaf_type, StringType)
                ),
                None,
            )

        if chosen is None:
            source_df.select(root_col).printSchema()
            raise ValueError(
                f"Could not find a numeric or string field inside {root_col}."
            )

        path, leaf_type = chosen
        value_expr = F.col(root_col)

        for part in path:
            value_expr = value_expr.getField(part)

        print(f"Extracting {root_col} from nested field: {'.'.join(path)}")

        if isinstance(leaf_type, NumericType):
            return value_expr.cast("double")

        return extract_numeric_from_string(value_expr)

    source_df.select(root_col).printSchema()
    raise ValueError(f"Unsupported type for {root_col}: {data_type}")


# -----------------------------
# Resolve reel count
# -----------------------------

reel_root_col = next(
    (c for c in ["number_of_reels", "reels"] if c in source_columns),
    None,
)

if reel_root_col is None:
    raise ValueError("Could not find number_of_reels or reels.")

reel_expr = numeric_feature_expr(
    reel_root_col,
    [
        "number_of_reels",
        "reel_count",
        "count",
        "base_reels",
        "reels",
        "value",
        "score",
    ],
)


# -----------------------------
# Identify theme and mechanic features
# -----------------------------

theme_cols = [
    c for c in source_columns
    if c.startswith("theme_")
    and c.endswith("_score")
    and c != "theme_presence_score"
]

mechanic_present_cols = [
    c for c in source_columns
    if c.startswith("mechanic_") and c.endswith("_present")
]
mechanic_score_cols = [
    c for c in source_columns
    if c.startswith("mechanic_") and c.endswith("_score")
]

# Prefer binary mechanic presence features when available.
mechanic_cols = mechanic_present_cols or mechanic_score_cols

if not theme_cols:
    raise ValueError("No theme_*_score columns found.")
if not mechanic_cols:
    raise ValueError(
        "No mechanic_*_present or mechanic_*_score columns found."
    )


# -----------------------------
# Resolve persistence feature(s)
# -----------------------------

persistence_candidates = [
    "persistence_score",
    "perceived_persistence_score",
    "persistence",
    "perceived_persistence",
]
persistence_cols = [
    c for c in persistence_candidates if c in source_columns
]

if not persistence_cols:
    raise ValueError("No recognized persistence feature found.")

persistence_exprs = {
    c: numeric_feature_expr(
        c,
        ["value", "score", "persistence", "persistence_score"],
    )
    for c in persistence_cols
}


# -----------------------------
# Resolve ways/paylines feature
# -----------------------------

ways_candidates = [
    "ways",
    "lines_or_ways",
    "paylines_or_ways_count",
    "lines",
]
ways_col = next(
    (c for c in ways_candidates if c in source_columns),
    None,
)

if ways_col is None:
    raise ValueError("No recognized ways/lines feature found.")

ways_expr = numeric_feature_expr(
    ways_col,
    [
        "value",
        "ways",
        "lines",
        "count",
        "paylines_or_ways_count",
        "score",
    ],
)


# -----------------------------
# Select source and matched-name columns
# -----------------------------

sales_name_col = (
    "to_sales_name_match"
    if "to_sales_name_match" in source_columns
    else "game_name"
)
rental_name_col = "to_rental_name_match"
eilers_name_col = (
    "to_eilers_ts_name_match"
    if "to_eilers_ts_name_match" in source_columns
    else "game_name"
)

selected_cols = [
    "game_name",
    "parent_game",
    "game_category",
    "join_game_name_key",
    OWNED_STATUS_COL,
    "parent_own_status",
    "game_classification",
    "max_months_since_release",
    'game_release_date',
    "has_sales_match",
    "has_rental_match",
    "has_eilers_ts_match",
    rental_name_col,
]

if sales_name_col != "game_name":
    selected_cols.append(sales_name_col)

if eilers_name_col != "game_name":
    selected_cols.append(eilers_name_col)


# -----------------------------
# Clean and unpack features
# -----------------------------

clean_df = source_df.select(
    *[F.col(c) for c in selected_cols],
    reel_expr.alias("reels_clean"),
    ways_expr.alias("ways_clean"),
    *[
        persistence_exprs[c].alias(c)
        for c in persistence_cols
    ],
    *[
        F.expr(f"try_cast(`{c}` AS DOUBLE)").alias(c)
        for c in theme_cols
    ],
    *[
        F.expr(f"try_cast(`{c}` AS DOUBLE)").alias(c)
        for c in mechanic_cols
    ],
)

# Standardize the ownership column name for the next cell.
if OWNED_STATUS_COL != "own_status":
    clean_df = clean_df.withColumnRenamed(
        OWNED_STATUS_COL,
        "own_status",
    )

# Fill missing theme and mechanic values with zero.
for c in theme_cols + mechanic_cols:
    clean_df = clean_df.withColumn(
        c,
        F.coalesce(F.col(c), F.lit(0.0)),
    )

# Normalize the four grouping keys. Null or blank values use "__unknown__".
grouping_cols = [
    #"game_category",
    #"own_status",
    "parent_own_status",
    #"game_classification",
]

for c in grouping_cols:
    clean_df = clean_df.withColumn(
        c,
        F.when(
            F.col(c).isNull()
            | (F.trim(F.col(c).cast("string")) == ""),
            F.lit("__unknown__"),
        ).otherwise(
            F.lower(F.trim(F.col(c).cast("string")))
        ),
    )

# Prepare the canonical game key; merge duplicate games below.
clean_df = (
    clean_df
    .withColumn(
        "_game_name_key",
        F.regexp_replace(F.lower(F.trim(F.col("join_game_name_key"))), r"\s+", " "),
    )
    .where(F.col("_game_name_key").isNotNull() & (F.col("_game_name_key") != ""))
)

pdf = clean_df.toPandas()

if len(pdf) < 2:
    raise ValueError("Fewer than two games remain after preprocessing.")


# -----------------------------
# Clean flags and numeric values
# -----------------------------

def as_bool(value):
    if pd.isna(value):
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


for c in ["has_sales_match", "has_rental_match", "has_eilers_ts_match"]:
    pdf[c] = pdf[c].map(as_bool)

# One row per canonical game, even if several overview documents exist.
# Keep one document's features and combine source matches across documents.
def normalize_game_reference(value):
    """Normalize exact game references; null-like strings are missing."""
    if value is None or pd.isna(value):
        return None
    key = " ".join(str(value).strip().lower().split())
    return None if key in {
        "", "none", "null", "<na>", "nan", "na", "n/a", "__unknown__"
    } else key


def deduplicate_games(frame):
    frame = frame.copy()
    frame["join_game_name_key"] = (
        frame["join_game_name_key"].astype("string").str.strip().str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )
    frame = frame.loc[
        frame["join_game_name_key"].notna() & frame["join_game_name_key"].ne("")
    ]
    # Prefer the alphabetically first document name as the feature row.
    frame = frame.sort_values("game_name", kind="stable", na_position="last")
    match_columns = [
        ("has_sales_match", sales_name_col),
        ("has_rental_match", rental_name_col),
        ("has_eilers_ts_match", eilers_name_col),
    ]
    merged = []
    for _, group in frame.groupby("join_game_name_key", sort=False):
        row = group.iloc[0].copy()
        # Retain a parent supplied by another document when the feature row
        # has no parent. Use the same stable document order as source matches.
        parent_names = group["parent_game"].map(normalize_game_reference).dropna()
        row["parent_game"] = None if parent_names.empty else parent_names.iloc[0]
        for flag_col, name_col in match_columns:
            row[flag_col] = bool(group[flag_col].any())
            names = group.loc[group[flag_col], name_col].astype("string").str.strip()
            usable = names.notna() & ~names.str.lower().isin(
                ["", "none", "null", "<na>", "nan", "na", "n/a"]
            )
            names = names.loc[usable]
            # A false match flag has no usable mapping for this source.
            row[name_col] = None if names.empty else str(names.iloc[0])
        merged.append(row)
    if not merged:
        return frame.iloc[:0].reset_index(drop=True)
    return pd.DataFrame(merged).reset_index(drop=True)


before_dedup = len(pdf)
pdf = deduplicate_games(pdf)
print(f"Merged {before_dedup - len(pdf)} duplicate game rows by join_game_name_key.")

feature_groups = {
    "mechanics": mechanic_cols,
    "themes": theme_cols,
    "reels": ["reels_clean"],
    "persistence": persistence_cols,
    "ways": ["ways_clean"],
    "months_since_release": ["max_months_since_release"],
}

numeric_cols = [
    c
    for columns in feature_groups.values()
    for c in columns
]

# Keep null release ages null for the earlier-than comparison.
release_age = pd.to_numeric(
    pdf["max_months_since_release"],
    errors="coerce",
)

# Coerce malformed values to null, then fill feature nulls with zero.
for c in numeric_cols:
    pdf[c] = pd.to_numeric(
        pdf[c],
        errors="coerce",
    ).fillna(0.0)

pdf["max_months_since_release"] = release_age


# -----------------------------
# Min-max scale features to [0, 1]
# -----------------------------

scaled_data = {}

for c in numeric_cols:
    values = pdf[c].astype(float)
    lo, hi = values.min(), values.max()

    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        scaled_data[c] = np.zeros(len(pdf), dtype=float)
    else:
        scaled_data[c] = (
            (values - lo) / (hi - lo)
        ).to_numpy(dtype=float)

scaled_pdf = pd.DataFrame(scaled_data)


# -----------------------------
# Normalize feature-group weights
# -----------------------------

invalid_weights = {
    group: weight
    for group, weight in FEATURE_GROUP_WEIGHTS.items()
    if not isinstance(weight, (int, float)) or weight < 0
}

if invalid_weights:
    raise ValueError(
        f"Feature group weights must be non-negative numbers: {invalid_weights}"
    )

active_groups = {
    group: columns
    for group, columns in feature_groups.items()
    if columns and FEATURE_GROUP_WEIGHTS.get(group, 0) > 0
}

weight_total = sum(
    FEATURE_GROUP_WEIGHTS[group]
    for group in active_groups
)

if weight_total <= 0:
    raise ValueError("At least one feature group must have a positive weight.")

normalized_weights = {
    group: FEATURE_GROUP_WEIGHTS[group] / weight_total
    for group in active_groups
}


# -----------------------------
# Generate experiment name
# -----------------------------

def readable_slug(text, max_length=70):
    text = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        str(text),
    ).strip("_").lower()

    return text[:max_length].strip("_") or "ai_attribute_similarity"


def weight_based_name():
    top_groups = sorted(
        normalized_weights.items(),
        key=lambda item: item[1],
        reverse=True,
    )[:3]

    return f"weighted_{method}_" + "_".join(
        group for group, _ in top_groups
    )


if USER_EXPERIMENT_NAME:
    EXPERIMENT_READABLE_NAME = USER_EXPERIMENT_NAME
else:
    weights_text = ", ".join(
        f"{group}={weight:.2f}"
        for group, weight in normalized_weights.items()
    )

    prompt = (
        "Suggest one short readable name, 3 to 7 words, for a slot-game "
        f"similarity experiment using {method} distance and these feature "
        f"weights: {weights_text}. Return only the name."
    )

    try:
        escaped_prompt = prompt.replace("'", "''")
        suggestion = spark.sql(
            "SELECT ai_query("
            f"'{AI_NAME_MODEL}', '{escaped_prompt}'"
            ") AS suggested_name"
        ).first()["suggested_name"]

        EXPERIMENT_READABLE_NAME = (
            str(suggestion).strip().strip('"').strip("'")
        )

        if not EXPERIMENT_READABLE_NAME:
            EXPERIMENT_READABLE_NAME = weight_based_name()

    except Exception as e:
        EXPERIMENT_READABLE_NAME = weight_based_name()
        print(
            "AI name suggestion unavailable; using a weight-based name. "
            f"Details: {str(e)[:250]}"
        )

EXPERIMENT_READABLE_NAME = readable_slug(
    EXPERIMENT_READABLE_NAME
)

print(f"Experiment name: {EXPERIMENT_READABLE_NAME}")
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Parent game process: {PARENT_GAME_PROCESS}")
print(f"Distance method: {method}")
print(f"Normalized feature weights: {normalized_weights}")
print(f"Games prepared: {len(pdf)}")

In [0]:
display(pdf)

In [0]:
import numpy as np
import pandas as pd

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    DoubleType,
    ArrayType,
    MapType,
    DateType,
)


# --------------------------------------------------
# Prepare data and scale features to [0, 1]
# --------------------------------------------------
pdf = pdf.reset_index(drop=True)

feature_columns = list(dict.fromkeys(
    column
    for columns in active_groups.values()
    for column in columns
))

# Grouping columns are intentionally not required. They do not restrict
# neighbor matching, and their output values can be null until added later.
required_columns = [
    "join_game_name_key",
    "parent_game",
    "max_months_since_release",
    "game_release_date",
    "to_sales_name_match",
    "to_rental_name_match",
    "has_sales_match",
    "has_rental_match",
    "has_eilers_ts_match",
    sales_name_col,
    rental_name_col,
    eilers_name_col,
    *feature_columns,
]

missing_columns = sorted(set(required_columns) - set(pdf.columns))
if missing_columns:
    raise ValueError(f"Missing required columns in pdf: {missing_columns}")

if "analysis_split" in pdf.columns:
    fit_mask = (
        pdf["analysis_split"]
        .astype("string")
        .str.lower()
        .eq("train")
        .fillna(False)
    )
    if not fit_mask.any():
        raise ValueError("No training rows found to fit the feature scaler.")
else:
    # This cell has no train/test split, so fit on all games.
    fit_mask = pd.Series(True, index=pdf.index)

# Convert feature values to numeric.
raw_features = pdf[feature_columns].apply(pd.to_numeric, errors="coerce")

# Missing theme/mechanic values are zero.
zero_fill_columns = [
    column for column in feature_columns
    if column.startswith(("theme_", "mechanic_", "mech_"))
]
if zero_fill_columns:
    raw_features[zero_fill_columns] = raw_features[zero_fill_columns].fillna(0.0)

# Impute remaining missing values with medians fitted on the scaler rows.
other_columns = [
    column for column in feature_columns
    if column not in zero_fill_columns
]
if other_columns:
    fit_medians = raw_features.loc[fit_mask, other_columns].median()
    raw_features[other_columns] = raw_features[other_columns].fillna(fit_medians)

# Fit min-max bounds and scale each feature to [0, 1].
feature_min = raw_features.loc[fit_mask, feature_columns].min()
feature_max = raw_features.loc[fit_mask, feature_columns].max()
feature_range = feature_max - feature_min

scaled_pdf = (
    (raw_features - feature_min)
    / feature_range.replace(0, np.nan)
).fillna(0.0).clip(0.0, 1.0)

game_records = pdf.to_dict("records")
game_names = pdf["join_game_name_key"].fillna("").astype(str).to_numpy()

candidate_ages = pd.to_numeric(
    pdf["max_months_since_release"],
    errors="coerce",
).to_numpy(dtype=np.float64, na_value=np.nan)
valid_ages = ~np.isnan(candidate_ages)

candidate_release_dates = pd.to_datetime(
    pdf["game_release_date"],
    errors="coerce",
).to_numpy(dtype="datetime64[ns]")


# --------------------------------------------------
# Match-flag and name helpers
# --------------------------------------------------
def has_valid_name(value):
    """Return whether a value is a usable matched source name."""
    if pd.isna(value):
        return False

    return str(value).strip().lower() not in {
        "",
        "none",
        "null",
        "<na>",
        "nan",
        "na",
        "n/a",
    }


def to_bool(value):
    """Safely interpret boolean or string match flags."""
    if pd.isna(value):
        return False

    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes", "y"}

    return bool(value)


def optional_string(record, column):
    """Return a string value when present; otherwise return None."""
    value = record.get(column)

    if value is None or pd.isna(value):
        return None

    return str(value)


has_sales_match = (
    pdf["has_sales_match"]
    .map(to_bool)
    .to_numpy(dtype=bool)
)
has_rental_match = (
    pdf["has_rental_match"]
    .map(to_bool)
    .to_numpy(dtype=bool)
)
has_eilers_match = (
    pdf["has_eilers_ts_match"]
    .map(to_bool)
    .to_numpy(dtype=bool)
)

valid_sales_names = (
    pdf["to_sales_name_match"].map(has_valid_name).to_numpy(dtype=bool)
    & pdf[sales_name_col].map(has_valid_name).to_numpy(dtype=bool)
)
valid_rental_names = (
    pdf[rental_name_col].map(has_valid_name).to_numpy(dtype=bool)
)
valid_eilers_names = (
    pdf[eilers_name_col].map(has_valid_name).to_numpy(dtype=bool)
)


# Final target eligibility is based on the source match flags.
# Keep all source rows in pdf so existing scaling and Eilers candidate
# coverage are preserved. Only eligible targets become output rows.
eligible_targets = (
    has_eilers_match
    & (has_sales_match | has_rental_match)
)


def sales_rental_type(index):
    if has_sales_match[index] and has_rental_match[index]:
        return "both"
    if has_sales_match[index]:
        return "sales"
    if has_rental_match[index]:
        return "rental"
    return None



# --------------------------------------------------
# Resolve parent references without fuzzy matching
# --------------------------------------------------
parent_keys = pdf["parent_game"].map(normalize_game_reference).to_numpy()
has_parent_game = np.array([key is not None for key in parent_keys], dtype=bool)
canonical_lookup = {
    normalize_game_reference(name): index
    for index, name in enumerate(game_names)
}

# Prefer the canonical key. Otherwise accept a unique exact matched-name
# alias; ambiguous aliases are reported and never guessed.
parent_alias_lookup = {}
for index, record in enumerate(game_records):
    for column in dict.fromkeys([sales_name_col, rental_name_col, eilers_name_col]):
        alias = normalize_game_reference(record.get(column))
        if alias is not None:
            parent_alias_lookup.setdefault(alias, set()).add(index)


def resolve_parent_index(target_idx):
    parent_key = parent_keys[target_idx]
    if parent_key is None:
        return None, "no_parent"
    if parent_key in canonical_lookup:
        parent_idx = canonical_lookup[parent_key]
    else:
        matches = parent_alias_lookup.get(parent_key, set())
        if not matches:
            return None, "not_found"
        if len(matches) != 1:
            return None, "ambiguous_name"
        parent_idx = next(iter(matches))
    if parent_idx == target_idx:
        return None, "self_reference"
    return int(parent_idx), "resolved"


parent_resolutions = [resolve_parent_index(i) for i in range(len(game_records))]


# --------------------------------------------------
# Build one weighted vector from all active features
# --------------------------------------------------
# Split each group's weight across its features so groups with many
# columns do not get extra influence just because they are larger.
weighted_parts = []

for group, columns in active_groups.items():
    group_weight = float(normalized_weights[group])
    feature_weight = np.sqrt(group_weight / len(columns))

    group_values = scaled_pdf[columns].to_numpy(
        dtype=np.float64,
        copy=True,
    )
    weighted_parts.append(group_values * feature_weight)

# Shape: (number of games, number of active features)
feature_matrix = np.concatenate(weighted_parts, axis=1)
feature_norms = np.linalg.norm(feature_matrix, axis=1)


# --------------------------------------------------
# Calculate one cosine distance across the full vector
# --------------------------------------------------
def weighted_distances_to(target_idx, candidate_indices):
    """Calculate full-vector cosine distances to eligible candidates."""
    target_vector = feature_matrix[target_idx]
    candidate_vectors = feature_matrix[candidate_indices]

    denominators = (
        feature_norms[target_idx]
        * feature_norms[candidate_indices]
    )

    dot_products = np.einsum(
        "ij,j->i",
        candidate_vectors,
        target_vector,
    )

    similarities = np.zeros(
        len(candidate_indices),
        dtype=np.float64,
    )

    np.divide(
        dot_products,
        denominators,
        out=similarities,
        where=denominators != 0,
    )

    # If both vectors are all zero, treat them as identical.
    zero_denominator = denominators == 0
    if np.any(zero_denominator):
        target_is_zero = np.isclose(feature_norms[target_idx], 0.0)
        candidate_is_zero = np.isclose(
            feature_norms[candidate_indices[zero_denominator]],
            0.0,
        )

        both_zero = target_is_zero & candidate_is_zero
        similarities[zero_denominator] = np.where(
            both_zero,
            1.0,
            0.0,
        )

    similarities = np.clip(similarities, -1.0, 1.0)
    return 1.0 - similarities

def rank_candidates(target_idx, candidate_mask):
    """Calculate and sort eligible candidates for one target."""
    candidate_indices = np.flatnonzero(candidate_mask)

    if len(candidate_indices) == 0:
        return []

    candidate_distances = weighted_distances_to(
        target_idx,
        candidate_indices,
    )

    # Sort by distance, then game name to break ties.
    sort_order = np.lexsort((
        game_names[candidate_indices],
        candidate_distances,
    ))

    return list(zip(
        candidate_indices[sort_order],
        candidate_distances[sort_order],
    ))


def make_neighbor_list(ranked_candidates, matched_name_col, limit):
    """Build a neighbor list from validated and ranked candidates."""
    neighbors = []
    if limit <= 0:
        return neighbors

    for candidate_idx, distance in ranked_candidates:
        candidate = game_records[int(candidate_idx)]
        matched_name = candidate[matched_name_col]

        neighbors.append({
            "join_game_name_key": str(candidate["join_game_name_key"]),
            "matched_source_name": str(matched_name).strip(),
            "similarity_score": float(max(0.0, 1.0 - distance)),
            "distance": float(distance),
        })

        if len(neighbors) >= limit:
            break

    return neighbors


# --------------------------------------------------
# Find Sales, Rental, and Eilers neighbors independently
# --------------------------------------------------
neighbor_cutoff = np.datetime64(pd.Timestamp(MAX_NEIGHBOUR_DATE))
all_indices = np.arange(len(game_records))


def candidate_masks_for(target_idx):
    """Share production eligibility rules with the single-game preview."""
    empty = np.zeros(len(game_records), dtype=bool)
    if not eligible_targets[target_idx] or not valid_ages[target_idx]:
        return {"sales": empty.copy(), "rental": empty.copy(), "eilers": empty.copy()}

    # Every neighbor must have a usable Eilers match, plus the existing
    # age, date-cutoff, and self-exclusion requirements.
    base_eligible = (
        has_eilers_match
        & valid_eilers_names
        & valid_ages
        & (candidate_ages >= candidate_ages[target_idx] + MIN_AGE_GAP_MONTHS)
        & (all_indices != target_idx)
        & (candidate_release_dates > neighbor_cutoff)
    )
    # P1 only restricts targets that have a parent reference. Candidates
    # need a parent of their own; it does not have to be the same parent.
    if PARENT_GAME_PROCESS == "p1_parent_games_only" and has_parent_game[target_idx]:
        base_eligible &= has_parent_game

    return {
        "sales": base_eligible & has_sales_match & valid_sales_names,
        "rental": base_eligible & has_rental_match & valid_rental_names,
        # Eilers-only games can still supply performance neighbors.
        "eilers": base_eligible,
    }


def ranked_candidates_for(target_idx, source, candidate_mask):
    """Apply the selected process; production and preview share this list."""
    ranked = rank_candidates(target_idx, candidate_mask)
    if (
        PARENT_GAME_PROCESS != "p3_include_parent"
        or source not in {"sales", "rental"}
        or not eligible_targets[target_idx]
    ):
        return ranked

    parent_idx, _ = parent_resolutions[target_idx]
    if parent_idx is None:
        return ranked
    available = (
        has_sales_match[parent_idx] and valid_sales_names[parent_idx]
        if source == "sales"
        else has_rental_match[parent_idx] and valid_rental_names[parent_idx]
    )
    if not available:
        return ranked

    # Explicit parent inclusion is an exception to the usual candidate
    # age-gap, release cutoff and Eilers requirements. The parent must be
    # a distinct, uniquely resolved game with a usable commercial match.
    # Reserve one of the configured N slots, keeping its real distance.
    parent_distance = float(weighted_distances_to(
        target_idx, np.array([parent_idx], dtype=int)
    )[0])
    return [(parent_idx, parent_distance)] + [
        (index, distance) for index, distance in ranked if int(index) != parent_idx
    ]


def parent_is_in_list(target_idx, neighbors):
    parent_idx, _ = parent_resolutions[target_idx]
    return parent_idx is not None and any(
        item["join_game_name_key"] == game_names[parent_idx] for item in neighbors
    )


result_rows = []

for target_idx in np.flatnonzero(eligible_targets):
    target_idx = int(target_idx)
    target = game_records[target_idx]
    masks = candidate_masks_for(target_idx)

    # Compute all three lists for each eligible target. Match flags describe
    # the target's actual source availability, independently of these lists.
    sales_neighbors = make_neighbor_list(
        ranked_candidates_for(target_idx, "sales", masks["sales"]),
        matched_name_col=sales_name_col,
        limit=N_SALES_NEIGHBORS,
    )
    rental_neighbors = make_neighbor_list(
        ranked_candidates_for(target_idx, "rental", masks["rental"]),
        matched_name_col=rental_name_col,
        limit=N_RENTAL_NEIGHBORS,
    )
    eilers_neighbors = make_neighbor_list(
        ranked_candidates_for(target_idx, "eilers", masks["eilers"]),
        matched_name_col=eilers_name_col,
        limit=N_EILERS_NEIGHBORS,
    )

    target_release_date = pd.to_datetime(
        target.get("game_release_date"),
        errors="coerce",
    )

    parent_idx, parent_status = parent_resolutions[target_idx]
    result_rows.append({
        "parent_game": parent_keys[target_idx],
        "parent_game_process": PARENT_GAME_PROCESS,
        "parent_game_match_status": parent_status,
        "resolved_parent_game_name": (
            None if parent_idx is None else str(game_names[parent_idx])
        ),
        "parent_in_sales_neighbors": parent_is_in_list(target_idx, sales_neighbors),
        "parent_in_rental_neighbors": parent_is_in_list(target_idx, rental_neighbors),
        "join_game_name": optional_string(target, "join_game_name_key"),
        "to_sales_name_match": optional_string(
            target,
            "to_sales_name_match",
        ),
        "to_rental_name_match": optional_string(target, rental_name_col),
        "to_eilers_ts_name_match": optional_string(target, eilers_name_col),
        "has_sales_match": bool(has_sales_match[target_idx]),
        "has_rental_match": bool(has_rental_match[target_idx]),
        "has_eilers_ts_match": bool(has_eilers_match[target_idx]),
        "sales_rental_match_type": sales_rental_type(target_idx),
        "game_category": optional_string(target, "game_category"),
        "own_status": optional_string(target, "own_status"),
        "parent_own_status": optional_string(target, "parent_own_status"),
        "game_classification": optional_string(
            target,
            "game_classification",
        ),
        "max_months_since_release": (
            None
            if pd.isna(target.get("max_months_since_release"))
            else float(target["max_months_since_release"])
        ),
        "sales_similar_games": sales_neighbors,
        "rental_similar_games": rental_neighbors,
        "eilers_similar_games": eilers_neighbors,
        "experiment_readable_name": EXPERIMENT_READABLE_NAME,
        "experiment_id": EXPERIMENT_ID,
        "distance_method": method,
        "game_release_date": (
            None
            if pd.isna(target_release_date)
            else target_release_date.date()
        ),
        "feature_group_weights": {
            group: float(weight)
            for group, weight in normalized_weights.items()
        },
    })


# --------------------------------------------------
# Create Spark output
# --------------------------------------------------
neighbor_schema = StructType([
    StructField("join_game_name_key", StringType(), True),
    StructField("matched_source_name", StringType(), True),
    StructField("similarity_score", DoubleType(), True),
    StructField("distance", DoubleType(), True),
])

output_schema = StructType([
    StructField("join_game_name", StringType(), True),
    StructField("parent_game", StringType(), True),
    StructField("parent_game_process", StringType(), False),
    StructField("parent_game_match_status", StringType(), False),
    StructField("resolved_parent_game_name", StringType(), True),
    StructField("parent_in_sales_neighbors", BooleanType(), False),
    StructField("parent_in_rental_neighbors", BooleanType(), False),
    StructField("to_sales_name_match", StringType(), True),
    StructField("to_rental_name_match", StringType(), True),
    StructField("to_eilers_ts_name_match", StringType(), True),
    StructField("has_sales_match", BooleanType(), False),
    StructField("has_rental_match", BooleanType(), False),
    StructField("has_eilers_ts_match", BooleanType(), False),
    StructField("sales_rental_match_type", StringType(), False),
    StructField("game_category", StringType(), True),
    StructField("own_status", StringType(), True),
    StructField("parent_own_status", StringType(), True),
    StructField("game_classification", StringType(), True),
    StructField("max_months_since_release", DoubleType(), True),
    StructField(
        "sales_similar_games",
        ArrayType(neighbor_schema),
        True,
    ),
    StructField(
        "rental_similar_games",
        ArrayType(neighbor_schema),
        True,
    ),
    StructField(
        "eilers_similar_games",
        ArrayType(neighbor_schema),
        True,
    ),
    StructField("experiment_readable_name", StringType(), True),
    StructField("experiment_id", StringType(), True),
    StructField("distance_method", StringType(), True),
    StructField("game_release_date", DateType(), True),
    StructField(
        "feature_group_weights",
        MapType(StringType(), DoubleType()),
        True,
    ),
])

like_games_ai_attr_df = spark.createDataFrame(
    result_rows,
    schema=output_schema,
)

print(f"Parent game process: {PARENT_GAME_PROCESS}")
print(
    "Parent resolution status:",
    pd.Series([row["parent_game_match_status"] for row in result_rows], dtype="string")
    .value_counts().to_dict(),
)
print(
    "Targets with parent in Sales / Rental lists:",
    sum(row["parent_in_sales_neighbors"] for row in result_rows),
    "/",
    sum(row["parent_in_rental_neighbors"] for row in result_rows),
)
print(f"Input games: {len(pdf)}")
print(f"Eligible output games: {len(result_rows)}")
print(f"Excluded without an Eilers match: {int((~has_eilers_match).sum())}")
print(
    "Excluded with Eilers but neither Sales nor Rental: "
    f"{int((has_eilers_match & ~has_sales_match & ~has_rental_match).sum())}"
)
print(
    "Eligible games by match type:",
    pd.Series([row["sales_rental_match_type"] for row in result_rows], dtype="string")
    .value_counts().to_dict(),
)
display(like_games_ai_attr_df)


In [0]:
target_name = "king khufu dawn"

# Use the same candidate masks as the production search above.
target_keys = (
    pdf["join_game_name_key"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)
target_key = " ".join(target_name.lower().split())
target_indices = np.flatnonzero(
    target_keys.eq(target_key).fillna(False).to_numpy()
)


def make_candidate_preview(target_idx, source, candidate_mask, matched_name_col, limit):
    rows = []
    for rank, (candidate_idx, distance) in enumerate(
        ranked_candidates_for(target_idx, source, candidate_mask), start=1
    ):
        candidate_idx = int(candidate_idx)
        candidate = game_records[candidate_idx]
        rows.append({
            "selection_order": rank,
            "selected_neighbor": rank <= max(0, limit),
            "parent_game": parent_keys[candidate_idx],
            "is_target_parent": candidate_idx == parent_resolutions[target_idx][0],
            "join_game_name_key": candidate["join_game_name_key"],
            "matched_source_name": candidate[matched_name_col],
            "has_sales_match": bool(has_sales_match[candidate_idx]),
            "has_rental_match": bool(has_rental_match[candidate_idx]),
            "game_release_date": candidate["game_release_date"],
            "max_months_since_release": candidate_ages[candidate_idx],
            "months_older_than_target": (
                candidate_ages[candidate_idx] - candidate_ages[target_idx]
            ),
            "distance": float(distance),
            "similarity_score": float(max(0.0, 1.0 - distance)),
        })
    return pd.DataFrame(rows)


# Clear previews on every run so a skipped target cannot leave stale results.
sales_close_games_df = pd.DataFrame()
rental_close_games_df = pd.DataFrame()
eilers_close_games_df = pd.DataFrame()

if len(target_indices) != 1:
    print(f"Preview skipped: found {len(target_indices)} matches for {target_name!r}.")
else:
    target_idx = int(target_indices[0])
    if not eligible_targets[target_idx]:
        print(
            f"{target_name!r} is excluded from the output: "
            "an Eilers match and at least one Sales/Rental match are required."
        )
    else:
        print(f"Target: {target_name}; match type: {sales_rental_type(target_idx)}")
        print(f"Parent process: {PARENT_GAME_PROCESS}")
        print(f"Parent: {parent_keys[target_idx]!r}; status: {parent_resolutions[target_idx][1]}")
        masks = candidate_masks_for(target_idx)
        sales_close_games_df = make_candidate_preview(
            target_idx, "sales", masks["sales"], sales_name_col, N_SALES_NEIGHBORS
        )
        rental_close_games_df = make_candidate_preview(
            target_idx, "rental", masks["rental"], rental_name_col, N_RENTAL_NEIGHBORS
        )
        eilers_close_games_df = make_candidate_preview(
            target_idx, "eilers", masks["eilers"], eilers_name_col, N_EILERS_NEIGHBORS
        )
        for label, preview in [
            ("Sales", sales_close_games_df),
            ("Rental", rental_close_games_df),
            ("Eilers", eilers_close_games_df),
        ]:
            print(f"{label} candidates in selection order for {target_name} (selected_neighbor marks the saved list):")
            if preview.empty:
                print("No candidates meet the neighbor requirements.")
            else:
                display(preview)


In [0]:
if OVERWRITE:
    like_games_ai_attr_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(like_games_dest)
else:
    like_games_ai_attr_df.write.mode("append").saveAsTable(like_games_dest)